# 26. EDA with pandas, seaborn & SQL

*Module: 6 — Python access · Companion summary: `lectures/26-eda-with-sql-and-pandas.md` · SQL engine: SQLite (in-memory)*

## What you will study

- profile a dataset with SQL + pandas describe()
- find extreme rows with idxmax / .at
- draw swarmplot, jointplot, boxplot — inline in this notebook

---
## Setup — build the sample database this notebook runs on
*Run the cell below first: it creates an in-memory SQLite database and the `q()` helper that prints every query's result set.*

## 🧪 Build the nutrition table in memory (a McDonald's-style menu)

Twelve items across 4 categories, sodium in mg.

In [ ]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(':memory:')

menu = pd.DataFrame({
    'item': ['Big Mac', 'Quarter W L Cheese', 'McChicken', 'French Fries L',
             'Chicken Mc N 20pc', 'Chicken Mc N 40pc', 'Sausage Mc Muffin',
             'Egg Mc Muffin', 'Hot Caramel Sundae', 'Apple Pie',
             'Garden Salad', 'Coca-Cola M'],
    'category': ['Beef', 'Beef', 'Chicken', 'Sides',
                 'Chicken', 'Chicken', 'Breakfast', 'Breakfast',
                 'Dessert', 'Dessert', 'Sides', 'Drinks'],
    'sodium_mg': [1010, 1290, 1200, 190,
                  1500, 3600, 780, 750, 150, 180, 65, 30],
})
menu.head()

## 🧪 SQL for the raw table + pandas describe() for statistics

In [ ]:
menu.to_sql('menu', conn, index=False, if_exists='replace')
pd.read_sql_query("SELECT category, COUNT(*) AS items, ROUND(AVG(sodium_mg),1) AS avg_sodium FROM menu GROUP BY category ORDER BY avg_sodium DESC", conn)

In [ ]:
menu[['sodium_mg']].describe()

## 🧪 Which item is the sodium bomb? idxmax + .at

In [ ]:
row = menu['sodium_mg'].idxmax()
print('max sodium row:', row)
print(menu.at[row, 'item'], '→', menu.at[row, 'sodium_mg'], 'mg')
menu[menu.category == 'Chicken'].sort_values('sodium_mg', ascending=False)

## 🧪 Plots (need `matplotlib inline` in a live session)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. swarmplot — every item's sodium, split by category
plt.figure(figsize=(7, 4))
sns.swarmplot(data=menu, x='category', y='sodium_mg', hue='category', legend=False)
plt.title('Sodium by category — swarmplot')
plt.tight_layout()
plt.show()  # displayed inline when running in Jupyter

In [ ]:
# 2. jointplot — sodium vs category index (relationship view)
plt.figure(figsize=(6, 4))
sns.jointplot(data=menu, x='sodium_mg', y='category', height=5)
plt.show()

In [ ]:
# 3. boxplot — distribution per category
plt.figure(figsize=(7, 4))
sns.boxplot(data=menu, x='category', y='sodium_mg', hue='category', legend=False)
plt.title('Sodium spread — boxplot')
plt.tight_layout()
plt.show()

## 💡 Tips & tricks

- SQL for aggregation, pandas/seaborn for the visual layer — combine both (course's
  lab does exactly this).
- 3600 mg sodium in the 40-piece McNuggets: >1.5× the daily allowance — EDA makes
  such outliers unmissable (the course's point!).

## 🎯 Your turn

1. Top-3 items by sodium (SQL ORDER BY + LIMIT — or pandas `.nlargest(3)`).
2. Average sodium per category as a seaborn barplot.

In [ ]:
# 💡 Solution
import pandas as pd
print(pd.read_sql_query("SELECT item, sodium_mg FROM menu ORDER BY sodium_mg DESC LIMIT 3", conn))
print(menu['sodium_mg'].nlargest(3))
plt.figure(figsize=(6, 3))
sns.barplot(data=menu, x='category', y='sodium_mg', hue='category', legend=False)
plt.title('Average sodium by category')
plt.tight_layout()
plt.show()